# Phân loại nhóm ngôn ngữ của tên bằng LSTM

Notebook chạy các module trong thư mục dự án. Dataset gồm họ/tên riêng; không suy ra quốc tịch thật của người mang tên. Đọc README để xem nguồn và hạn chế nhãn Wikidata.


In [ ]:
from pathlib import Path
import sys, json, subprocess
ROOT = Path.cwd()
assert (ROOT / "name_origin").is_dir(), "Mở notebook tại thư mục dự án"
print("Python:", sys.executable)


## 1. Dữ liệu và cách chia

Dữ liệu đã có sẵn. Không có tên chuẩn hóa hoặc nhóm phiên âm nào được dùng chung giữa các tập. Bản bỏ dấu chỉ bổ sung trong train.


In [ ]:
from name_origin.common import DATA, ARTIFACTS
stats = json.loads((DATA / "stats.json").read_text())
print(stats["split_counts"])
for label, values in stats["per_class"].items():
    print(label, values)


Nếu cần tạo lại dataset từ file raw đã tải, chạy cell sau. Chỉ chạy khi muốn tái tạo; dữ liệu hiện đã được chuẩn bị.


In [ ]:
# subprocess.run([sys.executable, "-m", "name_origin.prepare", "--offline"], check=True)


## 2. Kiến trúc LSTM

Embedding 32 chiều → LSTM 128 đơn vị → Dropout → lớp phân loại. Chọn trạng thái tại ký tự cuối thật, trước padding.


In [ ]:
import torch
from name_origin.model import NameLSTM
checkpoint = torch.load(ARTIFACTS / "model.pt", map_location="cpu", weights_only=True)
model = NameLSTM(len(checkpoint["vocab"]) + 2, len(checkpoint["labels"]), hidden_dim=checkpoint["hidden_dim"])
model.load_state_dict(checkpoint["state_dict"])
print(model)
print("Số tham số:", sum(p.numel() for p in model.parameters()))


## 3. Huấn luyện lại (tùy chọn)

Mất vài phút trên CPU. Chọn checkpoint bằng Macro-F1 validation; temperature và baseline cũng chỉ chọn theo validation.


In [ ]:
# subprocess.run([sys.executable, "-m", "name_origin.train"], check=True)


## 4. Kết quả thực tế

Xem cả Macro-F1 và kết quả bỏ dấu, thay vì chỉ accuracy tổng. Nhãn Wikidata là nhãn suy ra, cần test ngoài để xác nhận.


In [ ]:
report = json.loads((ARTIFACTS / "metrics.json").read_text())
for key in ["accuracy", "balanced_accuracy", "macro_f1", "top3_accuracy"]:
    print(key, "{:.2%}".format(report["test"][key]))
for label in checkpoint["labels"]:
    print(label, report["test"]["per_class"][label])
print("Test bỏ dấu:", report["test_all_accents_removed"]["accuracy"])


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ARTIFACTS / "confusion_matrix.png")))
display(Image(filename=str(ARTIFACTS / "training_curves.png")))


## 5. Nhập tên

Tên có nhiều nhãn như Lee sẽ được cảnh báo. Với họ tên đầy đủ, demo gộp trung bình điểm các thành phần; chưa có đánh giá riêng cho cách gộp.


In [ ]:
from name_origin.predict import Predictor
predictor = Predictor()
for name in ["Nguyễn", "Yamamoto", "Kim Min-jun", "John Smith", "Lee", "Nguyễn Đức Tuyên"]:
    result = predictor.predict(name)
    print(name, result["candidates"], result["warnings"])
